In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install face_recognition

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install dlib

Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
import tkinter as tk
from tkinter import messagebox, ttk
from PIL import Image, ImageTk
import cv2
import face_recognition
import csv
from historique_manager import HistoriqueManager

# Configuration des couleurs
PRIMARY_COLOR = "#3498db"
SECONDARY_COLOR = "#2980b9"
ACCENT_COLOR = "#e74c3c"
BG_COLOR = "#f5f7fa"
TEXT_COLOR = "#2c3e50"

# Dossier des visages connus
KNOWN_FACES_DIR = "face-detect"
known_face_encodings = []
known_face_names = []

def load_known_faces():
    """Charge les visages connus depuis le dossier"""
    for file_name in os.listdir(KNOWN_FACES_DIR):
        if file_name.endswith(".jpg"):
            image_path = os.path.join(KNOWN_FACES_DIR, file_name)
            image = face_recognition.load_image_file(image_path)
            encoding = face_recognition.face_encodings(image)[0]
            known_face_encodings.append(encoding)
            known_face_names.append(os.path.splitext(file_name)[0])

load_known_faces()
historique = HistoriqueManager()

def recognize_faces():
    """Fonction de reconnaissance faciale"""
    video_capture = cv2.VideoCapture(0)
    
    while True:
        ret, frame = video_capture.read()
        if not ret:
            print("Erreur lors de la capture vidéo.")
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_frame)
        face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
            name = "Inconnu"

            face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
            best_match_index = face_distances.argmin() if face_distances.size > 0 else -1

            if best_match_index != -1 and matches[best_match_index]:
                name = known_face_names[best_match_index]
                historique.ajouter_entree(name)

            color = (0, 255, 0) if name != "Inconnu" else (0, 0, 255)
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.putText(frame, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

        cv2.imshow("Reconnaissance faciale - Appuyez sur Q pour quitter", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    cv2.destroyAllWindows()

def start_recognition():
    """Lance la reconnaissance faciale"""
    if not known_face_encodings:
        messagebox.showerror("Erreur", "Aucune base de données de visages disponible.")
    else:
        recognize_faces()

def afficher_historique_fenetre():
    """Affiche l'historique dans une nouvelle fenêtre"""
    if not os.path.exists("historique.csv"):
        messagebox.showinfo("Info", "Aucun historique trouvé.")
        return

    fenetre_historique = tk.Toplevel(window)
    fenetre_historique.title("Historique des Entrées")
    fenetre_historique.geometry("800x500")
    fenetre_historique.configure(bg=BG_COLOR)
    
    # Style pour le Treeview
    style = ttk.Style(fenetre_historique)
    style.theme_use("clam")
    style.configure("Treeview", 
                   background="#ffffff",
                   foreground=TEXT_COLOR,
                   rowheight=25,
                   fieldbackground="#ffffff",
                   font=('Helvetica', 10))
    style.map('Treeview', background=[('selected', PRIMARY_COLOR)])

    # Cadre pour le Treeview
    frame_tree = ttk.Frame(fenetre_historique)
    frame_tree.pack(pady=20, padx=20, fill="both", expand=True)

    # Barre de défilement
    scrollbar = ttk.Scrollbar(frame_tree)
    scrollbar.pack(side="right", fill="y")

    tree = ttk.Treeview(frame_tree, columns=("Nom", "Date", "Heure"), show="headings", yscrollcommand=scrollbar.set)
    tree.heading("Nom", text="Nom", anchor="w")
    tree.heading("Date", text="Date", anchor="w")
    tree.heading("Heure", text="Heure", anchor="w")
    tree.column("Nom", width=250)
    tree.column("Date", width=250)
    tree.column("Heure", width=250)

    with open("historique.csv", "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # ignorer l'en-tête
        for row in reader:
            tree.insert("", tk.END, values=row)

    tree.pack(fill="both", expand=True)
    scrollbar.config(command=tree.yview)

    # Bouton de fermeture
    btn_close = ttk.Button(fenetre_historique, text="Fermer", command=fenetre_historique.destroy)
    btn_close.pack(pady=10)

# Création de la fenêtre principale
window = tk.Tk()
window.title("Système de Reconnaissance Faciale")
window.geometry("500x600")
window.configure(bg=BG_COLOR)

# Configuration du style global
style = ttk.Style()
style.theme_use("clam")
style.configure("TButton", 
               font=('Helvetica', 12, 'bold'),
               padding=10,
               background=PRIMARY_COLOR,
               foreground="white")
style.map("TButton",
          background=[('active', SECONDARY_COLOR)],
          foreground=[('active', 'white')])

# Logo et titre
logo_frame = tk.Frame(window, bg=BG_COLOR)
logo_frame.pack(pady=20)

try:
    logo_img = Image.open("logo_ucad.JPEG")
    logo_img = logo_img.resize((100, 100), Image.LANCZOS)
    logo_photo = ImageTk.PhotoImage(logo_img)
    logo_label = tk.Label(logo_frame, image=logo_photo, bg=BG_COLOR)
    logo_label.pack()
except FileNotFoundError:
    print("Logo non trouvé")

title_label = tk.Label(window, 
                      text="Reconnaissance Faciale",
                      font=('Helvetica', 20, 'bold'),
                      bg=BG_COLOR,
                      fg=TEXT_COLOR)
title_label.pack(pady=10)

subtitle_label = tk.Label(window,
                         text="Système d'identification des étudiants",
                         font=('Helvetica', 12),
                         bg=BG_COLOR,
                         fg=TEXT_COLOR)
subtitle_label.pack(pady=5)

# Boutons principaux
button_frame = tk.Frame(window, bg=BG_COLOR)
button_frame.pack(pady=30)

btn_start = ttk.Button(button_frame, 
                      text="Démarrer la Reconnaissance", 
                      command=start_recognition)
btn_start.pack(pady=15, ipadx=20)

btn_history = ttk.Button(button_frame,
                        text="Consulter l'Historique",
                        command=afficher_historique_fenetre)
btn_history.pack(pady=15, ipadx=20)

# Pied de page
footer_label = tk.Label(window,
                       text="© 2023 UCAD - Tous droits réservés",
                       font=('Helvetica', 8),
                       bg=BG_COLOR,
                       fg=TEXT_COLOR)
footer_label.pack(side="bottom", pady=10)

window.mainloop()